In [ ]:
from ax.service.utils.instantiation import ObjectiveProperties
from scipy.integrate import cumulative_trapezoid
from ax.api.configs import RangeParameterConfig
from ax.service.ax_client import AxClient
from dataclasses import dataclass, field
from ax.api.client import Client
import scipy.signal as signal
from pathlib import Path
import pandas as pd
import numpy as np
import subprocess
import shutil
import psutil
import time
import os
import re

current_path = Path(os.getcwd()).parent

In [ ]:
@dataclass
class DynaConfig:
    baseline_dir: Path = current_path / "baseline"
    bo_dir: Path = current_path / "bo-runs"
    lsdyna_exe: str = (
        r"C:\LSDYNA\program\ls-dyna_smp_s_R16.0_619-gd222963e2f_winx64_ifort190_sse2.exe"
    )
    input_file: str = "head.key"
    ncpu: int = 20
    memory: str = "1500m"
    timeout_hours: float = 2
    poll_interval: int = 10
    tests: list = field(default_factory=lambda: ["drop", "pendulum"])
    cfc_freq: int = 1000
    sampling_rate: int = 10000


def dyna_value(val):
    if isinstance(val, int):
        s = str(val)
        if len(s) > 10:
            s = f"{val:.3E}"
        return f"{s:<{10}}"
    abs_val = abs(val)
    if 1e-3 <= abs_val < 1e5:
        for decimals in range(6, -1, -1):
            s = f"{val:.{decimals}f}"
            if len(s) <= 10:
                return f"{s:<{10}}"
    for decimals in range(4, -1, -1):
        s = f"{val:.{decimals}E}"
        if len(s) <= 10:
            return f"{s:<{10}}"


def patch_params(params_path, parameters):
    with open(params_path, "r") as f:
        lines = f.readlines()
    for i in range(5, 8):
        sect = lines[i].strip().split()
        dtype = sect[0]
        param_name = sect[1]
        if param_name in parameters:
            value = parameters[param_name]
            sect0 = f"{dtype:<1}"
            sect1 = f"{param_name:<8}"[:8]
            value_dyna = dyna_value(value)
            lines[i] = f"{sect0} {sect1}{value_dyna}\n"
    with open(params_path, "w") as f:
        f.writelines(lines)


def initialize_iters(
    run_id,
    parameters,
    cfg,
):
    baseline_dir = Path(cfg.baseline_dir)
    bo_dir = Path(cfg.bo_dir)
    bo_dir.mkdir(parents=True, exist_ok=True)
    run_dir = bo_dir / f"run{run_id}"

    if run_dir.exists():
        shutil.rmtree(run_dir)
    shutil.copytree(baseline_dir, run_dir)
    print(f"[SPAWNED] [{run_id}] {run_dir}")

    for test in cfg.tests:
        patch_params(run_dir / test / "params", parameters)
        print(f"[PATCHED] [{run_id}] [{test}] {run_dir/test/"params"}")


def dyna_exec(run_id, test, cfg):
    test_path = Path(cfg.bo_dir) / Path("run" + str(run_id)) / test
    bat_path = test_path / "launchdyna.bat"
    bat_content = f"""@echo off
cd /d "{test_path}"
"{cfg.lsdyna_exe}" i={cfg.input_file} ncpu={cfg.ncpu} memory={cfg.memory}

"""
    with open(bat_path, "w") as f:
        f.write(bat_content)
    # print(f"[LAUNCH]  [{run_id}] [{test}] {bat_path}")
    return bat_path


def inspect_termination(test_path):
    d3hsp = Path(test_path) / "d3hsp"
    if not d3hsp.exists():
        print("[ERROR] d3hsp missing")
        return False
    try:
        with open(d3hsp, "r", errors="ignore") as f:
            lines = f.read()
        return "N o r m a l    t e r m i n a t i o n" in lines
    except Exception:
        return False


def killer(pid):
    parent = psutil.Process(pid)
    children = parent.children(recursive=True)
    for child in children:
        child.kill()
    parent.kill()


def launch_run(run_id, cfg):
    bo_dir = Path(cfg.bo_dir)
    run_dir = bo_dir / f"run{run_id}"
    prcss = {}
    for test in cfg.tests:
        test_dir = run_dir / test
        bat_path = dyna_exec(run_id, test, cfg)
        print(f"[LAUNCH]  [{run_id}] [{test}] {test_dir}")
        p = subprocess.Popen(str(bat_path), cwd=test_dir, shell=True)
        prcss[test] = p
    start_time = time.time()
    try:
        while True:
            elapsed_hours = (time.time() - start_time) / 3600
            if elapsed_hours > cfg.timeout_hours:
                print(f"[TIMEOUT] [{run_id}] Exceeded {cfg.timeout_hours} hrs")
                for p in prcss.values():
                    killer(p.pid)
                raise TimeoutError(f"[ERROR] Run {run_id} exceeded timeout")
            done = all(p.poll() is not None for p in prcss.values())
            if done:
                checks = {
                    test: inspect_termination(run_dir / test) for test in cfg.tests
                }
                if all(checks.values()):
                    print(f"[SOLVED]  [{run_id}] Normal termination for all models")
                    return True
                failed = [test for test, ok in checks.items() if not ok]
                raise RuntimeError(f"[FAILED] [{run_id}] {failed} execution failed")
            time.sleep(cfg.poll_interval)
    except KeyboardInterrupt:
        print("[INTERRUPT] Killing LS-DYNA")
        for p in prcss.values():
            killer(p.pid)
        raise


def cfc(data, cfg):
    filter_order = 2
    cutoff_frequency = cfg.cfc_freq * 1.667
    nyquist = 0.5 * cfg.sampling_rate
    b, a = signal.butter(filter_order, cutoff_frequency / nyquist, btype="low")
    filtered_data = signal.filtfilt(b, a, data)
    return filtered_data


def hic15(df):
    t_sec = df["time"].values / 1000
    a_g = df["resultant"].values * (1000.0 / 9.80665)
    A = cumulative_trapezoid(a_g, t_sec, initial=0)

    max_hic = 0.0
    best_t1 = 0.0
    best_t2 = 0.0
    n = len(t_sec)

    for i in range(n):
        for j in range(i + 1, n):
            dt = t_sec[j] - t_sec[i]
            if dt > 0.015:  # change here for different time interval
                break
            if dt > 0:
                integral_val = A[j] - A[i]
                avg_accel = integral_val / dt
                hic_val = dt * (avg_accel**2.5)
                if hic_val > max_hic:
                    max_hic = hic_val
                    best_t1 = t_sec[i]
                    best_t2 = t_sec[j]
    return max_hic


def bric(df):
    wx_rad_s = df["wx"].abs().max() * 1000.0
    wy_rad_s = df["wy"].abs().max() * 1000.0
    wz_rad_s = df["wz"].abs().max() * 1000.0
    bric_val = np.sqrt(
        (wx_rad_s / 56.45) ** 2 + (wy_rad_s / 66.25) ** 2 + (wz_rad_s / 42.87) ** 2
    )

    # return {
    #     'BrIC': bric,
    #     'Model_X_Pitch_rad_s': wx_rad_s,
    #     'Model_Y_Roll_rad_s': wy_rad_s,
    #     'Model_Z_Yaw_rad_s': wz_rad_s
    # }
    return bric_val


def postprocess(run_id, cfg):
    bo_dir = Path(cfg.bo_dir)
    run_dir = bo_dir / f"run{run_id}"

    num_pattern = r"[+-]?\d+\.\d+E[+-]?\d+"
    time_pattern = r"at time\s*([+-]?\d+\.\d+E[+-]?\d+)"
    hic_drop = None
    mom_pend = None
    hic_pend = None
    bric_drop = None
    bric_pend = None

    for test in cfg.tests:
        accels = []
        rot_vel = []
        moments = []
        current_time = None

        with open(run_dir / test / "nodout") as f:
            nodout = f.readlines()

        for i, step in enumerate(nodout):
            time_match = re.search(time_pattern, step)
            if time_match:
                current_time = float(time_match.group(1))
            if "x-disp" in step:
                data_line = nodout[i + 1]
                nums = re.findall(num_pattern, data_line)
                if len(nums) >= 9 and current_time is not None:
                    nums = list(map(float, nums))
                    accels.append((current_time, *nums[6:9]))
            elif "x-rot" in step:
                data_line = nodout[i + 1]
                nums = re.findall(num_pattern, data_line)
                if len(nums) >= 9 and current_time is not None:
                    nums = list(map(float, nums))
                    rot_vel.append((current_time, *nums[3:6]))

        accel_df = pd.DataFrame(accels, columns=["time", "x", "y", "z"])
        rot_vel_df = pd.DataFrame(rot_vel, columns=["time", "wx", "wy", "wz"])

        accel_df["resultant"] = cfc(
            np.linalg.norm(accel_df[["x", "y", "z"]].values, axis=1), cfg=cfg
        )
        rot_vel_df["wx"] = cfc(rot_vel_df["wx"], cfg=cfg)
        rot_vel_df["wy"] = cfc(rot_vel_df["wy"], cfg=cfg)
        rot_vel_df["wz"] = cfc(rot_vel_df["wz"], cfg=cfg)
        if test == "drop":
            hic_drop = hic15(accel_df)
            bric_drop = bric(rot_vel_df)
        if test == "pendulum":
            hic_pend = hic15(accel_df)
            bric_pend = bric(rot_vel_df)
            with open(run_dir / test / "secforc", "r") as f:
                secforc = f.readlines()
            for i, step in enumerate(secforc):
                if "ac ID =" in step:
                    time_line = secforc[i - 1]
                    time_nums = re.findall(num_pattern, time_line)
                    moment_nums = re.findall(num_pattern, step)
                    if len(time_nums) >= 1 and len(moment_nums) >= 4:
                        current_time = float(time_nums[0])
                        moment_nums = list(map(float, moment_nums))
                        moments.append((current_time, moment_nums[3]))
            moment_df = pd.DataFrame(moments, columns=["Time", "Resultant"])
            mom_pend = moment_df["Resultant"].abs().max()
    return hic_drop, mom_pend, hic_pend, bric_drop, bric_pend


def blackbox(run_id, parameters, **overrides):

    cfg = DynaConfig(**overrides)

    initialize_iters(run_id=run_id, parameters=parameters, cfg=cfg)
    launch_run(run_id=run_id, cfg=cfg)
    hic_drop, mom_pend, hic_pend, bric_drop, bric_pend = postprocess(
        run_id=run_id, cfg=cfg
    )

    if run_id == 0:
        master_df = pd.DataFrame(
            columns=[
                "run_id",
                "scale",
                "rotate",
                "slack",
                "hic_drop",
                "mom_pend",
                "hic_pend",
                "bric_drop",
                "bric_pend",
            ]
        )
        master_df.to_csv(cfg.bo_dir / "master.csv", index=False)
    master_df = pd.read_csv(cfg.bo_dir / "master.csv")
    new_row = {
        "run_id": run_id,
        "scale": parameters["scale"],
        "rotate": parameters["rotate"],
        "slack": parameters["slack"],
        "hic_drop": hic_drop,
        "mom_pend": mom_pend,
        "hic_pend": hic_pend,
        "bric_drop": bric_drop,
        "bric_pend": bric_pend,
    }
    master_df = pd.concat([master_df, pd.DataFrame([new_row])], ignore_index=True)
    master_df.to_csv(cfg.bo_dir / "master.csv", index=False)

    results = {
        "hic_drop": hic_drop,
        "mom_pend": mom_pend,
        "hic_pend": hic_pend,
        "bric_drop": bric_drop,
        "bric_pend": bric_pend,
    }
    return results

In [ ]:
client = Client()

parameters = [
    RangeParameterConfig(name="scale", parameter_type="float", bounds=(0.95, 1.1)),
    RangeParameterConfig(name="rotate", parameter_type="float", bounds=(-15.0, 15.0)),
    RangeParameterConfig(name="slack", parameter_type="float", bounds=(-15, 3)),
]

In [ ]:
client.configure_experiment(
    name="helmet_fit_bayesian_optimization", parameters=parameters
)

hic_pen_lim = 750.0
bric_drop_lim = 0.8
bric_pend_llim = 0.8

client.configure_optimization(
    objective="-hic_drop, -mom_pend",
    outcome_constraints=[
        f"hic_pend <= {hic_pen_lim}",
        f"bric_drop <= {bric_drop_lim}",
        f"bric_pend <= {bric_pend_llim}",
    ],
)

initial_guess = {"scale": 1.0, "rotate": 0.0, "slack": -1.0}

trial_index = client.attach_trial(parameters=initial_guess)

results = blackbox(parameters=initial_guess, run_id=trial_index)
client.complete_trial(trial_index=trial_index, raw_data=results)

max_iters = 50

for i in range(max_iters - 1):

    trials = client.get_next_trials(max_trials=1)

    for trial_index, params in trials.items():
        print(f"Launching optimization trial {trial_index}...")
        try:
            results = blackbox(run_id=trial_index, parameters=params)
            client.complete_trial(trial_index=trial_index, raw_data=results)

        except RuntimeError:
            client.experiment.trials[trial_index].mark_abandoned()
            print(f"Trial {trial_index} crashed. Trial abandoned.")
        except TimeoutError:
            client.experiment.trials[trial_index].mark_abandoned()
            print(f"Trial {trial_index} timed out. Trial abandoned.")

pareto_frontier = client.get_pareto_frontier()

print("\n--- Final Pareto Optimal Designs ---")
for params, metrics, trial_index, arm_name in pareto_frontier:
    print(f"\nTrial Index: {trial_index}")
    print(f"Optimal Parameters: {params}")
    print(f"Objectives/Constraints: {metrics}")

In [ ]:
cards = client.compute_analyses(display=True)